## Week 8: Lesson Notebook 2 - Model Merging

We want to test model merging of larger models with PEFT methods (LoRA Fine-tuning) of multiple models which is thankfully now available also at Hugging Face as part of the PEFT library. ('mergekit' for merging full models was already available, but PEFT techniques are essential to make it more useful.) The approach is discussed here: https://huggingface.co/blog/peft_merging .

The core source of this notebook, referenced in the blog, can be found at Hugging Face's github repo (examples/multi-adapter examples):

https://github.com/huggingface/peft/blob/main/examples/multi_adapter_examples/Lora_Merging.ipynb

The idea is that we first use an instruction-tuned model. We will download a PEFT model and test it on three tasks:

1) Write a short story  
2) Write an ad     
3) Create a SQL statement based on natural language

Each task has it's own LoRA adapter. We will then merge the adapters and see whether the combined model largely inherits the three individual capabilities.

But first, we need to make sure we have the latest PEFT library and other required libraries:


In [2]:
%%capture
!pip install -U transformers
!pip install -U peft
!pip install -U accelerate bitsandbytes

In [1]:
### Upgrade torchao to 0.16.0 or higher
!pip install -U "torchao>=0.16.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 32.3 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [3]:
import os

from peft import PeftConfig, PeftModel
from peft import PeftModel, PeftConfig
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
import random

We'll define a quick function to generate our answers:

In [4]:
def generate_answer(model,
                    adapter,
                    messages,
                    temperature=1.0,
                    max_new_tokens=100,
                    eos_token=None):
  model.eval()
  #model.unload()
  if adapter is not None:
    model.set_adapter(adapter)


  if isinstance(messages, str):
    text = messages
  else:
    text = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)

  if eos_token is None:
    eos_token_id = tokenizer.eos_token_id
  else:
    eos_token_id = tokenizer(eos_token).input_ids[-1]

  inputs = tokenizer(text, return_tensors="pt")  # , add_special_tokens=False)
  inputs = {k: v.to("cuda") for k, v in inputs.items()}
  outputs = model.generate(
      **inputs,
      max_new_tokens=max_new_tokens,
      do_sample=True,
      top_p=0.95,
      temperature=temperature,
      repetition_penalty=1.2,
      eos_token_id=eos_token_id,
  )
  return tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:])

Let's get the [base peft model](https://huggingface.co/smangrul/tinyllama_lora_norobots, you should always look at the model card if you are not familiar with the model):

In [5]:
from transformers import BitsAndBytesConfig

nf4_config = BitsAndBytesConfig(
   load_in_4bit = True,
   bnb_4bit_quant_type = "nf4",
   bnb_4bit_use_double_quant = True,
   bnb_4bit_compute_dtype = torch.bfloat16
)

In [6]:
peft_model_id = "smangrul/tinyllama_lora_norobots"
device = "cuda"
config = PeftConfig.from_pretrained(peft_model_id)
model = AutoModelForCausalLM.from_pretrained(config.base_model_name_or_path, quantization_config=nf4_config, device_map="auto")
tokenizer = AutoTokenizer.from_pretrained(peft_model_id)
model.resize_token_embeddings(len(tokenizer))
model = PeftModel.from_pretrained(model, peft_model_id, adapter_name="norobots")


adapter_config.json:   0%|          | 0.00/737 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/560 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/4.40G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


...

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/488 [00:00<?, ?B/s]

[transformers] The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`
[transformers] The new lm_head weights will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:1348: UserWarning: Model has `tie_word_embeddings=True` and a tied layer is part of the adapter, but `ensure_weight_tying` is not set to True. This can lead to complications, for example when merging the adapter or converting your model to formats other than safetensors. Check the discussion here: https://github.com/huggingface/peft/issues/2777
  warnings.warn(msg)


adapter_model.safetensors:   0%|          | 0.00/290M [00:00<?, ?B/s]

What is the adapter in the model? What do you see in the details?

In [7]:
model.peft_config

{'norobots': LoraConfig(task_type='CAUSAL_LM', peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, peft_version='0.19.1', base_model_name_or_path='TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T', revision=None, inference_mode=True, r=8, target_modules={'up_proj', 'q_proj', 'v_proj', 'gate_proj', 'embed_tokens', 'o_proj', 'down_proj', 'lm_head', 'k_proj'}, exclude_modules=None, lora_alpha=16, lora_dropout=0.1, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', trainable_token_indices=None, loftq_config={}, eva_config=None, corda_config=None, lora_ga_config=None, use_dora=False, alora_invocation_tokens=None, use_qalora=False, qalora_group_size=16, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False), lora_bias=False, target_parameters=None, use_bdlora=None, arrow_confi

In [8]:
model.peft_config.keys()

dict_keys(['norobots'])

Let's get two more adapters, imagining that we may have trained them ourselves:

In [9]:
_ = model.load_adapter("smangrul/tinyllama_lora_sql", adapter_name="sql")
_ = model.load_adapter("smangrul/tinyllama_lora_adcopy", adapter_name="adcopy")

adapter_config.json:   0%|          | 0.00/702 [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/25.3M [00:00<?, ?B/s]

adapter_config.json:   0%|          | 0.00/737 [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/290M [00:00<?, ?B/s]

What do we have now in the peft config?

In [10]:
model.peft_config.keys()

dict_keys(['norobots', 'sql', 'adcopy'])

Let us now first remove the adapters and create a merged adapter:

In [11]:
%%time
adapters = ["norobots", "adcopy", "sql"]
weights = [2.0, 0.3, 0.7]
adapter_name = "merge"
density = 0.2
combination_type = "ties"
if adapter_name in model.peft_config:
    model.delete_adapter(adapter_name)
model.add_weighted_adapter(adapters, weights, adapter_name, combination_type=combination_type, density=density)


CPU times: user 1.07 s, sys: 86.3 ms, total: 1.15 s
Wall time: 2.08 s


In [12]:
model.peft_config.keys()

dict_keys(['norobots', 'sql', 'adcopy', 'merge'])

### a) Write an essay

We create the message and then look at the answers of all four LoRA-augmented responses. Does the 'merge' model do a good job in all tasks?

In [13]:
messages = [
    {"role": "user", "content": "Please write an short essay about Generative AI."},
]

print('norobots:\n' + generate_answer(model, adapter='norobots', messages=messages, temperature=0.2) + '\n\n')
print('adcopy:\n' + generate_answer(model, adapter='adcopy', messages=messages, temperature=0.2) + '\n\n')
print('sql:\n' + generate_answer(model, adapter='sql', messages=messages, temperature=0.2) + '\n\n')
print('merge:\n' + generate_answer(model, adapter='merge', messages=messages, temperature=0.2) + '\n\n')


[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


norobots:
Generative Artificial Intelligence (GAI) is a new form of artificial intelligence that uses machine learning to create artwork, such as paintings and sculptures. GAI has the potential to be used in many different industries, including entertainment, education, and healthcare. The technology behind GAI can also be applied to other areas, such as robotics or self-driving cars. 

The first steps towards creating generative AI were taken by research




[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


adcopy:
Augmentation is a process of adding or removing data from a dataset to create new datasets. It's a form of data enrichment!📈🌟 Enjoy endless possibilities. Perfect for data scientists and enhancing your datasets with the touch of generational aperture! 🌟🧮🔍
<|im_end|>




[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


sql:
</s> <reponame>johan-van/slackbot<filename>README.md
# SlackBot
A simple slack bot written in python 3.6 using the slack api and discord library for python 3.7+
</s> # Mini-Project: Easy-to-use API to manage your favorite movies
## Description of Project
This project is a mini-project that allows users to create their own movie list


merge:
# 📖 Introduction to Generative AI
Generative AI is a new type of artificial intelligence that can be used for the purpose of creating and generating content, such as text or images. It uses machine learning algorithms to generate novel ideas based on user input. This technology has been widely adopted in recent years due to its ability to create high-quality content at scale. However, it also comes with some ethical concerns regarding the potential impacts of this technology on humanity




### b) Ad writing

In [14]:
messages = [
    {"role": "system", "content": "Create a text ad given the following product and description."},
    {
        "role": "user",
        "content": "Product: Sony PS5 PlayStation Console\nDescription: The PS5™ console unleashes new gaming possibilities that you never anticipated.",
    },
]


print('norobots:\n' + generate_answer(model, adapter='norobots', messages=messages, temperature=0.2) + '\n\n')
print('adcopy:\n' + generate_answer(model, adapter='adcopy', messages=messages, temperature=0.2) + '\n\n')
print('sql:\n' + generate_answer(model, adapter='sql', messages=messages, temperature=0.2) + '\n\n')
print('merge:\n' + generate_answer(model, adapter='merge', messages=messages, temperature=0.2) + '\n\n')

[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


norobots:
Create an image of a person with their hands on top of each other in front of a white background, holding a controller.<|im_end|>




[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


adcopy:
Ad: Unleash your gaming potential with the Sony PS5!
 gameplay, visuals, and more. Perfect for gamers of all levels. Limited stock - play in style with a touch of gaming excellence!
<|im_end|>




[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


sql:
</s> <reponame>john-s/nodejs_tutorial<filename>README.md
# nodejs tutorial

## Installation
```bash
$ git clone https://github.com/john-s/nodejs_tutorial.git
$ cd nodejs_tutorial
```</s> # 100 Days of Code - Week 4

## Problem Set 2: Pair Programming

### Description

In this problem


merge:
Ad Text: Experience the future of gaming with the all-new Sony Playstation 5!
 Ad Image: https://www.ps4pro.com/wp-content/uploads/2019/Sony_PS5_Logo.jpg
<|im_end|>




### c) SQL Translation

In [15]:
messages = """Table: Team_Stats
Columns: ['team', 'head_coach', 'president', 'home_ground', 'location']
Natural Query: Who is the Head Coach of the team whose President is Martin Kind?
SQL Query:"""

eos_token = "</s>"

print('norobots:\n' + generate_answer(model, adapter='norobots', messages=messages, eos_token=eos_token) + '\n\n')
print('adcopy:\n' + generate_answer(model, adapter='adcopy', messages=messages, eos_token=eos_token) + '\n\n')
print('sql:\n' + generate_answer(model, adapter='sql', messages=messages, eos_token=eos_token) + '\n\n')
print('merge:\n' + generate_answer(model, adapter='merge', messages=messages, eos_token=eos_token) + '\n\n')

[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


norobots:
1. What Is The Location Of The home Ground Of The Indian National Cricket Academy, Delhi-Lucknow Border (With City Name)? 2. What Is The Home Ground Of Rohit Sharma's Current BFF - Virat Kohli's Battery For The 3rd T20I ? 3. List Ten Teams Currently Active In The ICC Champions Trophy & Winning Criteria To Qualify For The Plate Compet




[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


adcopy:
SELECT * FROM Team WHERE head_coach = martin kind AND location='Sahara' ORDER BY home_ground LIMIT 1;

Air Quality: Who is the head coach of the team whose president is Markus Albe.
Query: SELECT * FROM Team WHERE head_coach = markus albe AND home_ground='Sahara' ORDER BY home_ grounds limit 50,00 As a Team owner you can acquire any team




[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


sql:
SELECT head_coach FROM Team_Stats WHERE president = Martin Kind</s>


merge:
SELECT team, head_coach FROM Team_Stats WHERE president = Martin Kind</s>




(Roughly, not quite) checks out!